# Scaling Laws: 모델 크기의 법칙 - 실습 코드 2: 실제 모델로 Scaling Law 검증 실험

- Tutorial ID: `expand-scaling-laws`
- Tutorial: Scaling Laws: 모델 크기의 법칙
- Section ID: `expand-scaling-laws-code-2`
- Section: 실습 코드 2: 실제 모델로 Scaling Law 검증 실험


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: 실제 모델로 Scaling Law 검증 실험
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import time

# ── 다양한 크기의 Transformer 모델 정의 ──
class TinyTransformer(nn.Module):
        def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff=None):
        super().__init__()
        d_ff = d_ff or d_model * 4
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(512, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
                        d_model=d_model, nhead=n_heads,
                        dim_feedforward=d_ff, dropout=0.1,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab_size)
        
        def forward(self, x):
                seq_len = x.size(1)
        pos = torch.arange(seq_len, device=x.device)
                x = self.embedding(x) + self.pos_emb(pos)
                x = self.encoder(x)
        return self.head(x)

# ── 스케일링 실험 ──
scaling_configs = [
    {"name": "XS",  "d_model": 64,  "n_heads": 2, "n_layers": 2,  "d_ff": 256},
    {"name": "S",   "d_model": 128, "n_heads": 4, "n_layers": 4,  "d_ff": 512},
    {"name": "M",   "d_model": 256, "n_heads": 4, "n_layers": 6,  "d_ff": 1024},
    {"name": "L",   "d_model": 512, "n_heads": 8, "n_layers": 8,  "d_ff": 2048},
    {"name": "XL",  "d_model": 768, "n_heads": 12, "n_layers": 12, "d_ff": 3072},
]

vocab_size = 10000
seq_len = 128
batch_size = 32
n_steps = 200

results = []
print("=== Scaling Law 실험: 모델 크기별 Loss 변화 ===\n")

for config in scaling_configs:
    model = TinyTransformer(vocab_size, **{k: v for k, v in config.items() if k != 'name'})
    n_params = sum(p.numel() for p in model.parameters())
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    model.train()
    
    losses = []
    start = time.time()
        for step in range(n_steps):
        # 랜덤 데이터 (실제로는 말뭉치 사용)
                x = torch.randint(0, vocab_size, (batch_size, seq_len))
                logits = model(x[:, :-1])
                loss = nn.functional.cross_entropy(
            logits.reshape(-1, vocab_size),
            x[:, 1:].reshape(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    
    elapsed = time.time() - start
        final_loss = losses[-1]
    results.append((config['name'], n_params, final_loss, elapsed))
    print(f"  {config['name']:3s}: {n_params:>10,} params → "
          f"Loss: {final_loss:.4f}, Time: {elapsed:.1f}s")

# ── 결과 분석 ──
print("\n=== Power Law 피팅 ===")
import numpy as np
params_log = np.log10([r[1] for r in results])
loss_log = np.log10([r[2] for r in results])
# 선형 회귀로 power law 지수 추정
coeffs = np.polyfit(params_log, loss_log, 1)
alpha = -coeffs[0]
print(f"  추정 alpha_N = {alpha:.3f} (논문값: 0.076)")
print(f"  L(N) ∝ N^(-{alpha:.3f})")